In [1]:
import pandas as pd
import os
from io import StringIO
import numpy as np

In [2]:
# Function to read a file with a specified delimiter
def read_file(path, sep, dtype=None):
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return None
    try:
        df = pd.read_csv(path, sep=sep, dtype=dtype)
        return df
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

In [3]:
# Load all four datasets
file1 = read_file('Data/Fahrzeug/Fahrzeuge_OEM1_Typ11.csv', sep=',')
file2 = read_file('Data/Fahrzeug/Fahrzeuge_OEM1_Typ12.csv', sep=';')
file3 = read_file('Data/Fahrzeug/Fahrzeuge_OEM2_Typ21.csv', sep=',')
file4 = read_file('Data/Fahrzeug/Fahrzeuge_OEM2_Typ22.csv', sep=';')

In [4]:
#Clean and Align the Data

data = [file1,file2, file3, file4]
columns_to_drop = ['Produktionsdatum_Origin_01011970', 'origin', 'Fehlerhaft_Fahrleistung', 'Produktionsdatum']

# Drop unnecessary columns in all files, including Produktionsdatum
# Ensure data types are consistent (convert date columns)
for i in range(len(data)):
    data[i] = data[i].drop(columns=columns_to_drop, errors='ignore')
    data[i]['Fehlerhaft_Datum'] = pd.to_datetime(data[i]['Fehlerhaft_Datum'], errors='coerce')

file1, file2, file3, file4 = data

In [5]:
#  Combine the Datasets
combined_data = pd.concat([file1, file2,file3,file4], ignore_index=True)

#  Filter the Data for 2014-2016
combined_data_filtered = combined_data[
    (combined_data['Fehlerhaft_Datum'].dt.year >= 2014) & 
    (combined_data['Fehlerhaft_Datum'].dt.year <= 2016)
]

In [6]:
# Identify and Present Defect Vehicles
# Filter to include only defective vehicles
defective_vehicles = combined_data_filtered[combined_data_filtered['Fehlerhaft'] == 1]

In [7]:
# Clean Up the DataFrame by Dropping Unnecessary Columns
# Drop the unnecessary columns (Unnamed and X1)
defective_vehicles = defective_vehicles.drop(columns=['Unnamed: 0', 'X1','Herstellernummer','Werksnummer','Fehlerhaft'], errors='ignore')

# Reset the index to remove the left-most index column
defective_vehicles.reset_index(drop=True, inplace=True)
# Display the first few rows of the cleaned defective vehicles data
print(len(defective_vehicles))
defective_vehicles.head()

120130


,ID_Fahrzeug,Fehlerhaft_Datum
0,11-1-12-371662,2014-01-01
1,11-1-12-371666,2014-01-01
2,11-1-12-371677,2014-01-01
3,11-1-12-371682,2014-01-01
4,11-1-12-371686,2014-01-01


In [8]:
# Save the final filtered dataset of defective vehicles to a CSV file
defective_vehicles.to_csv('cleaned_defective_vehicles_2014_2016.csv', index=False)

In [9]:
# Load the registration data
registrations = pd.read_csv('Data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv',delimiter=';')
registrations = registrations.drop(columns=['Unnamed: 0'])
# Display the first few rows to understand the structure
registrations.head()

,IDNummer,Gemeinden,Zulassung
0,11-1-11-1,DRESDEN,2009-01-01
1,11-1-11-2,DRESDEN,2009-01-01
2,12-1-12-1,LEIPZIG,2009-01-01
3,12-1-12-2,LEIPZIG,2009-01-01
4,12-1-12-3,DORTMUND,2009-01-01


In [10]:
# Filter the registration data to include only the IDs that are in the defective_vehicles DataFrame
filtered_registrations = registrations[registrations['IDNummer'].isin(defective_vehicles['ID_Fahrzeug'])]

# Display the filtered registrations to verify
filtered_registrations.head()

,IDNummer,Gemeinden,Zulassung
40443,21-2-21-2684,HEIDENAU1,2009-02-27
94391,22-2-22-7340,KNEITLINGEN,2009-04-01
98695,21-2-21-12045,GOSHEIM,2009-04-01
102937,21-2-21-12635,SCHLEUSEGRUND,2009-04-01
108530,21-2-21-13610,KOELN,2009-04-08


In [11]:
# List of workshop cities
workshop_cities =workshop_cities = ["AUGSBURG", "INGOLSTADT", "REGENSBURG", "WUERZBURG", "BAMBERG", 
                   "BAYREUTH", "ASCHAFFENBURG", "ERLANGEN", "ROSENHEIM", "LANDSHUT"]
# Filter the filtered_registrations data to include only these cities
filtered_registrations_by_location = filtered_registrations[filtered_registrations['Gemeinden'].isin(workshop_cities)]
# Reset the index to remove the left-most index column
filtered_registrations_by_location.reset_index(drop=True, inplace=True)

# save the final filtered data to a CSV file
filtered_registrations_by_location.to_csv('filtered_registrations_by_location.csv', index=False)

In [12]:
filtered_registrations_by_location.head()

,IDNummer,Gemeinden,Zulassung
0,21-2-21-88075,REGENSBURG,2010-07-30
1,21-2-21-126247,WUERZBURG,2011-03-11
2,12-1-12-181579,ASCHAFFENBURG,2012-08-27
3,12-1-12-183129,AUGSBURG,2012-09-17
4,12-1-12-183132,AUGSBURG,2012-09-17


In [13]:
#Defective_vehicles filtered by citys
defective_vehicles_loc = defective_vehicles[defective_vehicles['ID_Fahrzeug'].isin(filtered_registrations_by_location['IDNummer'])]
defective_vehicles_loc.to_csv('defective_vehicles_by_location.csv', index=False)
defective_vehicles_loc.head()

,ID_Fahrzeug,Fehlerhaft_Datum
182,11-1-11-562942,2014-01-03
183,11-1-11-562973,2014-01-03
198,11-1-12-372424,2014-01-04
199,11-1-12-372426,2014-01-04
200,11-1-12-372427,2014-01-04


In [14]:
#Merging with citys to have the whole data
merge_with_loc = defective_vehicles_loc.merge(filtered_registrations_by_location, left_on='ID_Fahrzeug', right_on='IDNummer', how='inner')
merge_with_loc=merge_with_loc.drop(columns=['IDNummer'])
merge_with_loc.head()

,ID_Fahrzeug,Fehlerhaft_Datum,Gemeinden,Zulassung
0,11-1-11-562942,2014-01-03,ASCHAFFENBURG,2012-11-21
1,11-1-11-562973,2014-01-03,ASCHAFFENBURG,2012-11-21
2,11-1-12-372424,2014-01-04,ERLANGEN,2012-11-21
3,11-1-12-372426,2014-01-04,ERLANGEN,2012-11-21
4,11-1-12-372427,2014-01-04,ERLANGEN,2012-11-21


In [15]:
# Now we will merge our final dataset containing defective vehicles with Bestandteile_Fahrzeug files on ID_Fahrzeug column.
# We will try to reach spare parts from vehichle to component and component to spare parts.
# To do that we will find out components that we need to look at.

bst_fahrzeug_11= read_file('Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv', ';')
bst_fahrzeug_12= read_file('Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv', ';')
bst_fahrzeug_21= read_file('Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv', ';')
bst_fahrzeug_22= read_file('Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv', ';')


In [16]:
#Merge all files

bst_fahrzeug_all = pd.concat([bst_fahrzeug_11, bst_fahrzeug_12, bst_fahrzeug_21, bst_fahrzeug_22], ignore_index=True)
bst_fahrzeug_all= bst_fahrzeug_all.drop(columns = ['Unnamed: 0', 'X1'])
bst_fahrzeug_all.head()

,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor,ID_Fahrzeug
0,K4-112-1121-3,K3SG1-105-1051-32,K2LE1-109-1091-2,K1BE1-101-1011-7,11-1-11-1
1,K4-112-1121-4,K3SG1-105-1051-141,K2ST1-109-1092-5,K1BE1-101-1011-12,11-1-11-2
2,K4-112-1121-7,K3SG1-105-1051-106,K2ST1-109-1092-57,K1BE1-101-1011-38,11-1-11-3
3,K4-112-1121-9,K3SG1-105-1051-21,K2ST1-109-1092-91,K1BE1-101-1011-97,11-1-11-4
4,K4-112-1121-11,K3SG1-105-1051-59,K2ST1-109-1092-4,K1BE1-101-1011-65,11-1-11-5


In [17]:
# Now merge with defective vehicles to filter the data for our needs

bst_defective = merge_with_loc.merge(bst_fahrzeug_all, on = 'ID_Fahrzeug', how= 'inner')
bst_defective.head()

,ID_Fahrzeug,Fehlerhaft_Datum,Gemeinden,Zulassung,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor
0,11-1-11-562942,2014-01-03,ASCHAFFENBURG,2012-11-21,K4-114-1141-146516,K3AG1-106-1061-111901,K2ST1-109-1092-905781,K1BE1-101-1011-169017
1,11-1-11-562973,2014-01-03,ASCHAFFENBURG,2012-11-21,K4-114-1141-146680,K3SG1-107-1071-541237,K2ST1-109-1092-906067,K1DI1-101-1041-224512
2,11-1-12-372424,2014-01-04,ERLANGEN,2012-11-21,K4-114-1141-147519,K3AG1-106-1061-111628,K2LE1-111-1111-35834,K1DI1-103-1031-224529
3,11-1-12-372426,2014-01-04,ERLANGEN,2012-11-21,K4-114-1141-147522,K3SG1-105-1051-360862,K2ST1-109-1092-906125,K1DI1-101-1041-224476
4,11-1-12-372427,2014-01-04,ERLANGEN,2012-11-21,K4-114-1141-147524,K3SG1-107-1071-541654,K2ST1-109-1092-905327,K1DI1-101-1041-224049


In [18]:
# Rename columns in bst_defective_motor before merging
bst_defectiveR=bst_defective.rename(columns={
    
    'Fehlerhaft_Datum': 'Fahrzeug_Fehlerhaft_Datum'
})
bst_defectiveR.head()

,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden,Zulassung,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor
0,11-1-11-562942,2014-01-03,ASCHAFFENBURG,2012-11-21,K4-114-1141-146516,K3AG1-106-1061-111901,K2ST1-109-1092-905781,K1BE1-101-1011-169017
1,11-1-11-562973,2014-01-03,ASCHAFFENBURG,2012-11-21,K4-114-1141-146680,K3SG1-107-1071-541237,K2ST1-109-1092-906067,K1DI1-101-1041-224512
2,11-1-12-372424,2014-01-04,ERLANGEN,2012-11-21,K4-114-1141-147519,K3AG1-106-1061-111628,K2LE1-111-1111-35834,K1DI1-103-1031-224529
3,11-1-12-372426,2014-01-04,ERLANGEN,2012-11-21,K4-114-1141-147522,K3SG1-105-1051-360862,K2ST1-109-1092-906125,K1DI1-101-1041-224476
4,11-1-12-372427,2014-01-04,ERLANGEN,2012-11-21,K4-114-1141-147524,K3SG1-107-1071-541654,K2ST1-109-1092-905327,K1DI1-101-1041-224049


In [19]:
#Split the dataframe according the parts of the vehicles so that we can use them later for defective spare parts analyis.
bst_defective1 = bst_defectiveR.copy()

bst_defective_karosserie = bst_defective1[['ID_Fahrzeug', 'Fahrzeug_Fehlerhaft_Datum', 'ID_Karosserie', 'Gemeinden']]
bst_defective_schaltung = bst_defective1[['ID_Fahrzeug', 'Fahrzeug_Fehlerhaft_Datum', 'ID_Schaltung', 'Gemeinden']]
bst_defective_sitze = bst_defective1[['ID_Fahrzeug', 'Fahrzeug_Fehlerhaft_Datum', 'ID_Sitze', 'Gemeinden']]
bst_defective_motor = bst_defective1[['ID_Fahrzeug', 'Fahrzeug_Fehlerhaft_Datum', 'ID_Motor', 'Gemeinden']]

In [20]:
#Inspect the data to ensure the structure
bst_defective_karosserie.head()

,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,ID_Karosserie,Gemeinden
0,11-1-11-562942,2014-01-03,K4-114-1141-146516,ASCHAFFENBURG
1,11-1-11-562973,2014-01-03,K4-114-1141-146680,ASCHAFFENBURG
2,11-1-12-372424,2014-01-04,K4-114-1141-147519,ERLANGEN
3,11-1-12-372426,2014-01-04,K4-114-1141-147522,ERLANGEN
4,11-1-12-372427,2014-01-04,K4-114-1141-147524,ERLANGEN


In [21]:
bst_defective_schaltung.head()

,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,ID_Schaltung,Gemeinden
0,11-1-11-562942,2014-01-03,K3AG1-106-1061-111901,ASCHAFFENBURG
1,11-1-11-562973,2014-01-03,K3SG1-107-1071-541237,ASCHAFFENBURG
2,11-1-12-372424,2014-01-04,K3AG1-106-1061-111628,ERLANGEN
3,11-1-12-372426,2014-01-04,K3SG1-105-1051-360862,ERLANGEN
4,11-1-12-372427,2014-01-04,K3SG1-107-1071-541654,ERLANGEN


In [22]:
bst_defective_sitze.head()

,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,ID_Sitze,Gemeinden
0,11-1-11-562942,2014-01-03,K2ST1-109-1092-905781,ASCHAFFENBURG
1,11-1-11-562973,2014-01-03,K2ST1-109-1092-906067,ASCHAFFENBURG
2,11-1-12-372424,2014-01-04,K2LE1-111-1111-35834,ERLANGEN
3,11-1-12-372426,2014-01-04,K2ST1-109-1092-906125,ERLANGEN
4,11-1-12-372427,2014-01-04,K2ST1-109-1092-905327,ERLANGEN


In [23]:
bst_defective_motor.head()

,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,ID_Motor,Gemeinden
0,11-1-11-562942,2014-01-03,K1BE1-101-1011-169017,ASCHAFFENBURG
1,11-1-11-562973,2014-01-03,K1DI1-101-1041-224512,ASCHAFFENBURG
2,11-1-12-372424,2014-01-04,K1DI1-103-1031-224529,ERLANGEN
3,11-1-12-372426,2014-01-04,K1DI1-101-1041-224476,ERLANGEN
4,11-1-12-372427,2014-01-04,K1DI1-101-1041-224049,ERLANGEN


In [24]:

# Function to clean duplicate columns by keeping only the .x columns and renaming them
def clean_duplicate_columns(df):
    # Identify columns that have a `.x` suffix
    columns_to_keep = [col for col in df.columns if col.endswith('.x')]
    
    # Create a mapping to rename these columns back to their original names
    rename_mapping = {col: col.replace('.x', '') for col in columns_to_keep}
    
    # Subset the DataFrame to keep only these columns
    df_cleaned = df[columns_to_keep].rename(columns=rename_mapping)
    
    return df_cleaned

In [25]:
def process_component_data(component_files, id_column, motor_data):
    all_components = []
    for file in component_files:
        # Load the component data
        component_data = pd.read_csv(file['path'], delimiter=file['delimiter'], dtype=str)
        
        # Clean up unnecessary columns
        component_data_cleaned = component_data.drop(
            columns=['Unnamed: 0', 'X1', 'Produktionsdatum', 'Produktionsdatum_Origin_01011970', 'origin'], 
            errors='ignore'
        )

        # Handle .x and .y columns
        if any(component_data_cleaned.columns.str.endswith('.x')):
            # Select only the columns that end with '.x'
            component_data_cleaned = component_data_cleaned.loc[:, component_data_cleaned.columns.str.endswith('.x')]
            # Rename the columns to remove the '.x' suffix
            component_data_cleaned.columns = component_data_cleaned.columns.str.replace('.x', '', regex=False)
        elif any(component_data_cleaned.columns.str.endswith('.y')):
            # If there are .y columns, drop them (they are assumed to be duplicate columns)
            component_data_cleaned = component_data_cleaned.drop(columns=component_data_cleaned.columns.str.endswith('.y'))
         # Remove unnecessary columns after handling .x/.y
        component_data_cleaned = component_data_cleaned.drop(
            columns=['Produktionsdatum', 'Herstellernummer', 'Werksnummer', 'Fehlerhaft_Fahrleistung'], 
            errors='ignore'
        )
        # Filter for defective components
        defective_components = component_data_cleaned[component_data_cleaned['Fehlerhaft'] == '1']
        
        # Rename columns to include the component name in the column headers
        defective_components_renamed = defective_components.rename(columns={
            'Fehlerhaft': f'{id_column}_Fehlerhaft',
            'Fehlerhaft_Datum': f'{id_column}_Fehlerhaftt_Datum'
        })
        
        # Merge with the motor data
        merged_data = pd.merge(defective_components_renamed, motor_data, on=id_column, how='inner')
        
        # Drop the duplicate 'Fehlerhaft' column if it exists
        merged_data = merged_data.drop(columns=[f'{id_column}_Fehlerhaft'])
        
        all_components.append(merged_data)
    
    # Concatenate all processed components
    return pd.concat(all_components, ignore_index=True)
# Due to the difficulties to convert the txt files into csv and the inablity to read them, our work will focus on the CSV files given for us.
# Example usage for motor components
motor_files = [
    {'path': 'Data/Komponente/Komponente_K1DI1.csv', 'delimiter': ','},
    {'path': 'Data/Komponente/Komponente_K1BE1.csv', 'delimiter': ','},
    {'path': 'Data/Komponente/Komponente_K1BE2.csv', 'delimiter': ';'}
]
final_motor_data = process_component_data(motor_files, 'ID_Motor', bst_defective_motor)
final_motor_data.head()


,ID_Motor,ID_Motor_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden
0,K1DI1-101-1041-224768,2013-12-23,11-1-12-372534,2014-01-04,REGENSBURG
1,K1DI1-101-1041-224508,2013-12-22,11-1-12-372551,2014-01-04,REGENSBURG
2,K1DI1-101-1041-225492,2013-12-28,11-1-12-374078,2014-01-10,BAMBERG
3,K1DI1-101-1041-226408,2014-01-03,11-1-11-568227,2014-01-16,ERLANGEN
4,K1DI1-101-1041-226939,2014-01-06,11-1-12-375999,2014-01-17,BAYREUTH


In [26]:
#Since the data was correctly processed and the discrepancy between the defect dates in the motor and vehicle data is present in the original data, it likely indicates that the different components were detected as defective at different times, which is not unusual.
#What This Means:
#1-Defect Detection Timing:
  #The motor defect (Fehlerhaft_Datum of 2013-11-10) was recorded earlier than the vehicle defect (Fehlerhaft_Datum of 2014-01-18). This suggests that the motor defect was identified first, and perhaps after further inspection or failure, the entire vehicle was later marked as defective.
#2-Independent Defects:
 #It's also possible that the motor defect and the vehicle defect were identified independently. For instance, the vehicle could have been flagged for another issue later on that required the entire vehicle to be marked as defective.


In [27]:
# Example usage for Sitze components
Sitze_files = [
    {'path': 'Data/Komponente/Komponente_K2ST2.csv', 'delimiter': ';'},
    
]
final_Sitze_data = process_component_data(Sitze_files, 'ID_Sitze', bst_defective_sitze)
final_Sitze_data.head()


,ID_Sitze,ID_Sitze_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden
0,K2ST2-109-1092-311080,2014-03-31,22-2-22-115093,2014-01-06,AUGSBURG
1,K2ST2-109-1092-315543,2014-04-20,22-2-22-116758,2014-01-26,LANDSHUT
2,K2ST2-109-1092-318176,2014-05-02,22-2-22-117791,2014-02-08,ROSENHEIM
3,K2ST2-109-1092-321276,2014-05-16,22-2-22-118947,2014-02-22,INGOLSTADT
4,K2ST2-109-1092-324881,2014-06-01,22-2-22-120230,2014-03-10,AUGSBURG


In [28]:
Karosserie_files=[
    {'path': 'Data/Komponente/Komponente_K4.csv', 'delimiter': ';'},
]
final_Karosserie_data = process_component_data(Karosserie_files, 'ID_Karosserie', bst_defective_karosserie)
final_Karosserie_data.head()

,ID_Karosserie,ID_Karosserie_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden


In [29]:
Schaltung_files = [
    
    {'path': 'Data/Komponente/Komponente_K3AG1.csv', 'delimiter': ','},
    {'path': 'Data/Komponente/Komponente_K3SG1.csv', 'delimiter': ','},
    {'path': 'Data/Komponente/Komponente_K3SG2.csv', 'delimiter': ','}
]
final_Schaltung_data = process_component_data(Schaltung_files, 'ID_Schaltung', bst_defective_schaltung)
final_Schaltung_data.head()

,ID_Schaltung,ID_Schaltung_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden
0,K3AG1-105-1051-66649,2013-01-09,11-1-11-565807,2014-01-10,AUGSBURG
1,K3AG1-105-1051-66660,2013-01-09,11-1-12-374078,2014-01-10,BAMBERG
2,K3AG1-105-1051-68553,2013-02-18,11-1-12-384772,2014-02-19,WUERZBURG
3,K3AG1-105-1051-70633,2013-04-02,11-1-11-599525,2014-04-04,AUGSBURG
4,K3AG1-105-1051-71185,2013-04-14,11-1-11-604378,2014-04-16,ERLANGEN


In [30]:
def process_component_parts(parts_files, id_column, final_component_data):
    all_parts_data = []
    
    for file in parts_files:
        # Extract the ID from the filename (you may adjust this depending on your file structure)
        component_id = file['component_id']
        
        # Load the parts data
        parts_data = pd.read_csv(file['path'], delimiter=file['delimiter'])
        
        # Rename the component ID column to the unified ID name
        parts_data = parts_data.rename(columns={component_id: id_column})
        
        # Append the cleaned parts data to the list
        all_parts_data.append(parts_data)
    
    # Combine all the parts data into a single DataFrame
    combined_parts_data = pd.concat(all_parts_data, ignore_index=True)
    
    # Reshape the data by pivoting based on the ID
    reshaped_parts_data = combined_parts_data.pivot_table(
        index=id_column, 
        values=[col for col in combined_parts_data.columns if col.startswith('ID_T')],
        aggfunc='first'  # Adjust as needed
    ).reset_index()
    
    # Merge with the final component data
    final_data = pd.merge(final_component_data, reshaped_parts_data, on=id_column, how='inner')
    
    return final_data



In [31]:
# Example Motor parts:
motor_parts_files = [
    {'path': 'Data/Komponente/Bestandteile_Komponente_K1BE1.csv', 'delimiter': ';', 'component_id': 'ID_K1BE1'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K1BE2.csv', 'delimiter': ';', 'component_id': 'ID_K1BE2'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K1DI1.csv', 'delimiter': ';', 'component_id': 'ID_K1DI1'}
]

final_motor_with_parts_data = process_component_parts(motor_parts_files, 'ID_Motor', final_motor_data)
final_motor_with_parts_data.head()

,ID_Motor,ID_Motor_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden,ID_T1,ID_T2,ID_T3,ID_T4,ID_T5,ID_T6,ID_T7,ID_T8
0,K1DI1-101-1041-224768,2013-12-23,11-1-12-372534,2014-01-04,REGENSBURG,1-201-2011-606373,2-202-2022-1062718,None,None,5-202-2012-281025,6-203-2032-337415,None,None
1,K1DI1-101-1041-224508,2013-12-22,11-1-12-372551,2014-01-04,REGENSBURG,1-202-2021-605159,2-202-2022-1061384,None,None,5-201-2012-281043,6-203-2032-336955,None,None
2,K1DI1-101-1041-225492,2013-12-28,11-1-12-374078,2014-01-10,BAMBERG,1-202-2021-607400,2-202-2022-1066153,None,None,5-202-2012-282414,6-204-2044-225968,None,None
3,K1DI1-101-1041-226408,2014-01-03,11-1-11-568227,2014-01-16,ERLANGEN,1-204-2044-305184,2-202-2022-1070309,None,None,5-201-2012-283358,6-203-2032-339829,None,None
4,K1DI1-101-1041-226939,2014-01-06,11-1-12-375999,2014-01-17,BAYREUTH,1-201-2011-611515,2-201-2011-460573,None,None,5-201-2012-284091,6-204-2044-227413,None,None


In [42]:
# Sitze Parts:
Sitze_parts_files = [
    {'path': 'Data/Komponente/Bestandteile_Komponente_K2LE1.csv', 'delimiter': ';', 'component_id': 'ID_K2LE1'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K2LE2.csv', 'delimiter': ';', 'component_id': 'ID_K2LE2'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K2ST1.csv', 'delimiter': ';', 'component_id': 'ID_K2ST1'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K2ST2.csv', 'delimiter': ';', 'component_id': 'ID_K2ST2'}
]

final_Sitze_with_parts_data = process_component_parts(Sitze_parts_files, 'ID_Sitze', final_Sitze_data)
final_Sitze_with_parts_data.head()

,ID_Sitze,ID_Sitze_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden,ID_T11,ID_T12,ID_T13,ID_T14,ID_T15,ID_T16,ID_T17,ID_T18,ID_T19,ID_T20
0,K2ST2-110-1101-156639,2017-02-23,21-2-21-430922,2016-03-10,ASCHAFFENBURG,None,None,None,None,None,16-212-2121-120865,17-215-2151-107535,18-209-2091-216474,None,None
1,K2ST2-110-1101-145204,2017-01-04,21-2-21-422230,2016-01-21,ASCHAFFENBURG,None,None,None,None,None,16-212-2121-118386,17-214-2141-212741,18-209-2091-211882,None,None
2,K2ST2-109-1092-311080,2014-03-31,22-2-22-115093,2014-01-06,AUGSBURG,None,None,None,None,None,16-212-2121-68463,17-215-2151-60893,18-211-2111-154208,None,None
3,K2ST2-110-1101-180318,2017-06-09,21-2-21-449693,2016-06-26,AUGSBURG,None,None,None,None,None,16-215-2151-176613,17-215-2151-112101,18-211-2111-284044,None,None
4,K2ST2-110-1101-156681,2017-02-24,21-2-21-431236,2016-03-12,AUGSBURG,None,None,None,None,None,16-212-2121-120934,17-209-2092-217340,18-215-2154-53514,None,None


In [33]:
# Karosserie has empty data after filtering, their parts will be irrelevant to our data
#Schaltung parts:
Schaltung_parts_files = [
    {'path': 'Data/Komponente/Bestandteile_Komponente_K3AG1.csv', 'delimiter': ';', 'component_id': 'ID_K3AG1'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K3AG2.csv', 'delimiter': ';', 'component_id': 'ID_K3AG2'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K3SG1.csv', 'delimiter': ';', 'component_id': 'ID_K3SG1'},
    {'path': 'Data/Komponente/Bestandteile_Komponente_K3SG2.csv', 'delimiter': ';', 'component_id': 'ID_K3SG2'}
]

final_Schaltung_with_parts_data = process_component_parts(Schaltung_parts_files, 'ID_Schaltung', final_Schaltung_data)
final_Schaltung_with_parts_data.head()


,ID_Schaltung,ID_Schaltung_Fehlerhaftt_Datum,ID_Fahrzeug,Fahrzeug_Fehlerhaft_Datum,Gemeinden,ID_T21,ID_T22,ID_T23,ID_T24,ID_T25,ID_T26,ID_T27
0,K3AG1-105-1051-66649,2013-01-09,11-1-11-565807,2014-01-10,AUGSBURG,21-208-2081-760266,None,None,24-206-2062-90459,25-206-2061-134703,None,None
1,K3AG1-105-1051-66660,2013-01-09,11-1-12-374078,2014-01-10,BAMBERG,21-208-2081-761279,None,None,24-208-2082-90381,25-206-2061-134731,None,None
2,K3AG1-105-1051-68553,2013-02-18,11-1-12-384772,2014-02-19,WUERZBURG,21-208-2081-783647,None,None,24-207-2071-61502,25-207-2072-92818,None,None
3,K3AG1-105-1051-70633,2013-04-02,11-1-11-599525,2014-04-04,AUGSBURG,21-208-2081-806049,None,None,24-206-2062-95723,25-206-2061-142745,None,None
4,K3AG1-105-1051-71185,2013-04-14,11-1-11-604378,2014-04-16,ERLANGEN,21-207-2071-227114,None,None,24-208-2082-96510,25-207-2072-96376,None,None


In [34]:

# Dataframe containing the counts of defective motors by cities
final_motor_data = final_motor_data.sort_values('Gemeinden')
counts_df = final_motor_data.groupby('Gemeinden').size().reset_index(name='Defective_Motors')

# Rename columns for clarity
counts_df.columns = ['Gemeinden', 'Defective_Motors']
#print(counts_df)

# Divide by 3 (3 Years) and round up, since we can't have a fraction of a defective vehicle
counts_df['Defective_Motors'] = np.ceil(counts_df['Defective_Motors'] / 3).astype(int)

# Print or inspect the result
print(counts_df)


       Gemeinden  Defective_Motors
0  ASCHAFFENBURG                 8
1       AUGSBURG                33
2        BAMBERG                12
3       BAYREUTH                10
4       ERLANGEN                13
5     INGOLSTADT                15
6       LANDSHUT                10
7     REGENSBURG                15
8      ROSENHEIM                 9
9      WUERZBURG                18


In [36]:
# Dataframe containing the counts of defective motors by cities
final_Schaltung_with_parts_data = final_Schaltung_with_parts_data.sort_values('Gemeinden')
counts_dfsitze = final_Schaltung_with_parts_data.groupby('Gemeinden').size().reset_index(name='Defective_Schaltungen')
# Divide by 3 (3 Years) and round up, since we can't have a fraction of a defective vehicle
counts_dfsitze['Defective_Schaltungen'] = np.ceil(counts_dfsitze['Defective_Schaltungen'] / 3).astype(int)

# Merge the two DataFrames
counts_df= pd.merge(counts_df, counts_dfsitze, on='Gemeinden', how='inner')
print(counts_df)



       Gemeinden  Defective_Motors  Defective_Schaltungen
0  ASCHAFFENBURG                 8                      5
1       AUGSBURG                33                     14
2        BAMBERG                12                      6
3       BAYREUTH                10                      4
4       ERLANGEN                13                      6
5     INGOLSTADT                15                      8
6       LANDSHUT                10                      7
7     REGENSBURG                15                      8
8      ROSENHEIM                 9                      5
9      WUERZBURG                18                      8


In [37]:
final_Sitze_data = final_Sitze_data.sort_values('Gemeinden')
counts_dfSitze = final_Sitze_data.groupby('Gemeinden').size().reset_index(name='Defective_Sitze')
# Divide by 3 (3 Years) and round up, since we can't have a fraction of a defective vehicle
counts_dfSitze['Defective_Sitze'] = np.ceil(counts_dfSitze['Defective_Sitze'] / 3).astype(int)
counts_df= pd.merge(counts_df, counts_dfSitze, on='Gemeinden', how='inner')
print(counts_df)
counts_df.to_csv('Final_dataset_group_05.csv', index=False)


       Gemeinden  Defective_Motors  Defective_Schaltungen  Defective_Sitze
0  ASCHAFFENBURG                 8                      5                1
1       AUGSBURG                33                     14                5
2        BAMBERG                12                      6                3
3       BAYREUTH                10                      4                1
4       ERLANGEN                13                      6                3
5     INGOLSTADT                15                      8                2
6       LANDSHUT                10                      7                2
7     REGENSBURG                15                      8                3
8      ROSENHEIM                 9                      5                1
9      WUERZBURG                18                      8                2
